# 03 Stock Reaction + Merged EDA

Compute post-earnings stock returns, join audio/text/market features, and run COGS108-style exploratory checks before modeling.

In [ ]:
from io import StringIO
from pathlib import Path
import time

import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

DEDUPED_AUDIO_PATH = PROCESSED_DIR / "audio_call_feature_table_deduped.csv"
TEXT_FEATURES_PATH = PROCESSED_DIR / "earnings_text_features.csv"
STOCK_RETURNS_OUTPUT = PROCESSED_DIR / "earnings_event_returns.csv"
ANALYSIS_OUTPUT = PROCESSED_DIR / "earnings_analysis_table.csv"
UNMATCHED_OUTPUT = PROCESSED_DIR / "earnings_analysis_unmatched_rows.csv"

RETURN_HORIZONS = [1, 5, 10]
DEDUPED_AUDIO_PATH, TEXT_FEATURES_PATH

In [ ]:
audio = pd.read_csv(DEDUPED_AUDIO_PATH)
text_features = pd.read_csv(TEXT_FEATURES_PATH)

audio["ticker"] = audio["ticker"].astype(str).str.upper().str.strip()
audio["call_date"] = pd.to_datetime(audio["call_date"], errors="coerce")
text_features["ticker"] = text_features["ticker"].astype(str).str.upper().str.strip()
text_features["call_date"] = pd.to_datetime(text_features["call_date"], errors="coerce")

audio["merge_key"] = audio["ticker"] + "_" + audio["call_date"].dt.strftime("%Y_%m_%d")
text_features["merge_key"] = text_features["ticker"] + "_" + text_features["call_date"].dt.strftime("%Y_%m_%d")

pd.DataFrame(
    {
        "table": ["audio", "text"],
        "rows": [len(audio), len(text_features)],
        "tickers": [audio["ticker"].nunique(), text_features["ticker"].nunique()],
        "missing_dates": [audio["call_date"].isna().sum(), text_features["call_date"].isna().sum()],
    }
)

In [ ]:
def fetch_yahoo_prices(ticker: str, start_date: pd.Timestamp, end_date: pd.Timestamp) -> pd.DataFrame:
    """Fetch daily closes from Yahoo's chart API without adding a yfinance dependency."""
    period1 = int(pd.Timestamp(start_date).tz_localize("UTC").timestamp())
    period2 = int(pd.Timestamp(end_date).tz_localize("UTC").timestamp())
    url = f"https://query2.finance.yahoo.com/v8/finance/chart/{ticker}"
    params = {"period1": period1, "period2": period2, "interval": "1d", "events": "history"}
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()
    payload = response.json()
    result = payload.get("chart", {}).get("result") or []
    if not result:
        return pd.DataFrame()

    chart = result[0]
    timestamps = chart.get("timestamp") or []
    quote = (chart.get("indicators", {}).get("quote") or [{}])[0]
    closes = quote.get("close") or []
    volumes = quote.get("volume") or []

    prices = pd.DataFrame(
        {
            "Date": pd.to_datetime(timestamps, unit="s", utc=True).tz_convert(None).normalize(),
            "Close": closes,
            "Volume": volumes if len(volumes) == len(timestamps) else np.nan,
        }
    )
    prices["ticker"] = ticker
    prices = prices.dropna(subset=["Date", "Close"]).sort_values("Date").reset_index(drop=True)
    return prices


def compute_event_returns(ticker: str, call_date: pd.Timestamp, prices: pd.DataFrame) -> dict[str, object]:
    row = {"ticker": ticker, "call_date": call_date, "stock_status": "ok"}
    if prices.empty:
        row["stock_status"] = "missing_price_data"
        return row

    prices = prices.sort_values("Date").reset_index(drop=True)
    pre_call = prices[prices["Date"] <= call_date]
    if pre_call.empty:
        row["stock_status"] = "no_pre_call_close"
        return row

    base_idx = int(pre_call.index[-1])
    base_close = float(prices.loc[base_idx, "Close"])
    row["pre_call_trading_date"] = prices.loc[base_idx, "Date"]
    row["pre_call_close"] = base_close

    for horizon in RETURN_HORIZONS:
        target_idx = base_idx + horizon
        if target_idx >= len(prices):
            row[f"return_{horizon}d_pct"] = np.nan
            row[f"price_date_{horizon}d"] = pd.NaT
            continue
        target_close = float(prices.loc[target_idx, "Close"])
        row[f"price_date_{horizon}d"] = prices.loc[target_idx, "Date"]
        row[f"return_{horizon}d_pct"] = (target_close / base_close - 1) * 100

    return row

In [ ]:
min_call_date = audio["call_date"].min() - pd.Timedelta(days=30)
max_call_date = audio["call_date"].max() + pd.Timedelta(days=45)
tickers = sorted(audio["ticker"].dropna().unique())

price_cache = {}
for ticker in tickers:
    try:
        price_cache[ticker] = fetch_yahoo_prices(ticker, min_call_date, max_call_date)
    except Exception as exc:  # Keep EDA resilient and auditable.
        print(f"Price fetch failed for {ticker}: {exc}")
        price_cache[ticker] = pd.DataFrame()
    time.sleep(0.05)

stock_rows = [
    compute_event_returns(row.ticker, row.call_date, price_cache.get(row.ticker, pd.DataFrame()))
    for row in audio[["ticker", "call_date"]].drop_duplicates().itertuples(index=False)
]
stock_returns = pd.DataFrame(stock_rows)
stock_returns["merge_key"] = stock_returns["ticker"] + "_" + pd.to_datetime(stock_returns["call_date"]).dt.strftime("%Y_%m_%d")

stock_returns.to_csv(STOCK_RETURNS_OUTPUT, index=False)
print(f"Wrote stock event percent returns: {STOCK_RETURNS_OUTPUT}")
stock_returns["stock_status"].value_counts(dropna=False)

In [ ]:
text_one = (
    text_features.sort_values(["merge_key", "word_count"], ascending=[True, False])
    .drop_duplicates("merge_key", keep="first")
)

stock_merge_cols = [
    "merge_key",
    "stock_status",
    "pre_call_trading_date",
    "pre_call_close",
]
for horizon in RETURN_HORIZONS:
    stock_merge_cols.extend([f"price_date_{horizon}d", f"return_{horizon}d_pct"])
stock_merge_cols = [column for column in stock_merge_cols if column in stock_returns.columns]

analysis = audio.merge(
    text_one.drop(columns=["ticker", "call_date"], errors="ignore"),
    on="merge_key",
    how="left",
    validate="one_to_one",
)
analysis = analysis.merge(
    stock_returns[stock_merge_cols],
    on="merge_key",
    how="left",
    validate="one_to_one",
)

analysis["has_text_match"] = analysis["word_count"].notna()
analysis["has_stock_match"] = analysis["stock_status"].eq("ok")

unmatched = analysis.loc[
    ~analysis["has_text_match"] | ~analysis["has_stock_match"],
    ["merge_key", "ticker", "call_date", "has_text_match", "has_stock_match", "stock_status"],
].copy()

analysis.to_csv(ANALYSIS_OUTPUT, index=False)
unmatched.to_csv(UNMATCHED_OUTPUT, index=False)

print(f"Wrote merged analysis table: {ANALYSIS_OUTPUT}")
print(f"Wrote unmatched audit rows: {UNMATCHED_OUTPUT}")
pd.DataFrame(
    {
        "metric": ["analysis_rows", "text_match_rate", "stock_ok_rate", "unmatched_rows"],
        "value": [
            len(analysis),
            analysis["has_text_match"].mean(),
            analysis["has_stock_match"].mean(),
            len(unmatched),
        ],
    }
)

In [ ]:
return_cols = [f"return_{horizon}d_pct" for horizon in RETURN_HORIZONS if f"return_{horizon}d_pct" in analysis.columns]
audio_index_cols = [
    "audio_stress_index",
    "audio_confidence_index",
    "audio_instability_index",
    "vocal_clarity_proxy",
]
text_index_cols = [
    "word_count",
    "negative_term_rate",
    "positive_term_rate",
    "finance_term_rate",
    "simple_sentiment_balance",
]
eda_cols = [column for column in audio_index_cols + text_index_cols + return_cols if column in analysis.columns]

analysis[eda_cols].describe().T

In [ ]:
correlation_cols = [column for column in eda_cols if analysis[column].notna().sum() >= 3]
correlation_matrix = analysis[correlation_cols].corr(numeric_only=True)

if return_cols:
    display(correlation_matrix[return_cols].sort_values(return_cols[0], ascending=False))
else:
    display(correlation_matrix)

In [ ]:
agg_spec = {
    "calls": ("merge_key", "count"),
    "text_matches": ("has_text_match", "sum"),
    "stock_matches": ("has_stock_match", "sum"),
    "avg_audio_stress": ("audio_stress_index", "mean"),
    "avg_sentiment_balance": ("simple_sentiment_balance", "mean"),
}
for horizon in RETURN_HORIZONS:
    col = f"return_{horizon}d_pct"
    if col in analysis.columns:
        agg_spec[f"avg_return_{horizon}d_pct"] = (col, "mean")

ticker_event_profile = (
    analysis.groupby("ticker", as_index=False)
    .agg(**agg_spec)
    .sort_values(["calls", "ticker"], ascending=[False, True])
)

ticker_event_profile.head(30)

In [ ]:
try:
    import matplotlib.pyplot as plt

    plot_cols = [column for column in return_cols + audio_index_cols + text_index_cols if column in analysis.columns]
    axes = analysis[plot_cols].hist(figsize=(14, 10), bins=30)
    plt.suptitle("Merged earnings-analysis feature distributions", y=1.02)
    plt.tight_layout()
except ImportError:
    print("matplotlib is not installed; skipping histogram plots.")